In [1]:
!pip install fastapi uvicorn pydantic httpx python-dotenv websockets

## MCP 标准工具实现

In [2]:
"""
MCP Server Implementation
遵循 Model Context Protocol 标准
"""
# 直接导入所需库
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Dict, List, Any, Optional, Union
import uuid
import json
from datetime import datetime
import asyncio

# 直接在当前文件定义所需的类
class ToolInputSchema(BaseModel):
    """工具输入模式定义"""
    type: str = "object"
    properties: Dict[str, Any] = Field(default_factory=dict)
    required: List[str] = Field(default_factory=list)

class ToolDefinition(BaseModel):
    """MCP 工具定义"""
    name: str
    description: str
    inputSchema: ToolInputSchema

class ToolCall(BaseModel):
    """工具调用请求"""
    id: str
    name: str
    arguments: Dict[str, Any]

class ToolResult(BaseModel):
    """工具执行结果"""
    id: str
    content: List[Dict[str, Any]]
    isError: bool = False
    errorMessage: Optional[str] = None

class MemoryItem(BaseModel):
    """短期记忆项"""
    id: str
    timestamp: str
    content: str
    type: str
    metadata: Dict[str, Any] = Field(default_factory=dict)

# 现在可以定义 MCP 服务器
class MCPServer:
    """MCP 协议服务器"""
    
    def __init__(self):
        self.app = FastAPI(title="Moodle AI Agent MCP Server")
        self.tools: Dict[str, callable] = {}
        self.tool_definitions: List[ToolDefinition] = []
        self.sessions: Dict[str, Dict] = {}
        
        self._setup_cors()
        self._setup_routes()
    
    def _setup_cors(self):
        """配置 CORS"""
        self.app.add_middleware(
            CORSMiddleware,
            allow_origins=["*"],
            allow_credentials=True,
            allow_methods=["*"],
            allow_headers=["*"],
        )
    
    def _setup_routes(self):
        """设置 API 路由"""
        
        @self.app.get("/health")
        async def health_check():
            return {
                "status": "healthy", 
                "timestamp": datetime.now().isoformat(),
                "tools_registered": len(self.tools)
            }
        
        @self.app.get("/tools")
        async def list_tools():
            """列出所有可用工具"""
            return {
                "tools": [tool.dict() for tool in self.tool_definitions]
            }
        
        @self.app.post("/tools/call")
        async def call_tool(tool_call: ToolCall):
            """调用指定工具"""
            if tool_call.name not in self.tools:
                raise HTTPException(
                    status_code=404, 
                    detail=f"Tool '{tool_call.name}' not found"
                )
            
            try:
                result = await self.tools[tool_call.name](**tool_call.arguments)
                return ToolResult(
                    id=tool_call.id,
                    content=[{"type": "text", "text": json.dumps(result, ensure_ascii=False)}],
                    isError=False
                ).dict()
            except Exception as e:
                return ToolResult(
                    id=tool_call.id,
                    content=[],
                    isError=True,
                    errorMessage=str(e)
                ).dict()
    
    def register_tool(self, name: str, func: callable, definition: ToolDefinition):
        """注册工具"""
        self.tools[name] = func
        self.tool_definitions.append(definition)
        print(f"✅ Tool registered: {name}")

# 创建服务器实例
mcp_server = MCPServer()
print("✅ MCP Server 创建成功！")

✅ MCP Server 创建成功！


## 短期记忆系统

In [3]:
"""
短期记忆系统
实现有效的上下文管理
"""
# 短期记忆系统 - 独立版本
# 实现有效的上下文管理

from typing import Dict, List, Any, Optional
from datetime import datetime
import uuid
import json
from collections import deque

# 如果之前没有定义这些类，在这里定义
try:
    from src.mcp.schema import MemoryItem, ConversationTurn
except ImportError:
    # 独立定义所需的类
    from pydantic import BaseModel, Field
    
    class MemoryItem(BaseModel):
        """短期记忆项"""
        id: str
        timestamp: str
        content: str
        type: str  # "user_message", "assistant_message", "tool_result"
        metadata: Dict[str, Any] = Field(default_factory=dict)
    
    class ConversationTurn(BaseModel):
        """对话轮次"""
        turn_id: str
        user_message: str
        assistant_message: Optional[str] = None
        tool_calls: List = Field(default_factory=list)
        tool_results: List = Field(default_factory=list)
        timestamp: str = Field(default_factory=lambda: datetime.now().isoformat())


class ShortTermMemory:
    """短期记忆管理器"""
    
    def __init__(self, max_size: int = 50, session_id: Optional[str] = None):
        self.session_id = session_id or str(uuid.uuid4())
        self.max_size = max_size
        self.memory_queue: deque = deque(maxlen=max_size)
        self.conversation_history: List[ConversationTurn] = []
        self.context_summary: str = ""
        self.created_at = datetime.now()
        self.last_accessed = datetime.now()
        self.user_preferences: Dict[str, Any] = {}
    
    def add_item(self, content: str, item_type: str, metadata: Dict[str, Any] = None) -> MemoryItem:
        """添加记忆项"""
        item = MemoryItem(
            id=str(uuid.uuid4()),
            timestamp=datetime.now().isoformat(),
            content=content,
            type=item_type,
            metadata=metadata or {}
        )
        
        self.memory_queue.append(item)
        self.last_accessed = datetime.now()
        
        # 更新上下文摘要
        self._update_context_summary()
        
        return item
    
    def add_user_message(self, message: str) -> MemoryItem:
        """添加用户消息"""
        return self.add_item(message, "user_message")
    
    def add_assistant_message(self, message: str, tool_calls: List = None) -> MemoryItem:
        """添加助手消息"""
        metadata = {"tool_calls": tool_calls} if tool_calls else {}
        return self.add_item(message, "assistant_message", metadata)
    
    def add_tool_result(self, tool_name: str, result: Dict) -> MemoryItem:
        """添加工具结果"""
        content = f"Tool '{tool_name}' returned: {json.dumps(result, ensure_ascii=False)[:500]}"
        return self.add_item(content, "tool_result", {"tool_name": tool_name, "result": result})
    
    def add_conversation_turn(
        self, 
        user_message: str, 
        assistant_message: str,
        tool_calls: List = None,
        tool_results: List = None
    ) -> ConversationTurn:
        """添加完整对话轮次"""
        turn = ConversationTurn(
            turn_id=str(uuid.uuid4()),
            user_message=user_message,
            assistant_message=assistant_message,
            tool_calls=tool_calls or [],
            tool_results=tool_results or []
        )
        
        self.conversation_history.append(turn)
        
        # 添加到记忆队列
        self.add_user_message(user_message)
        self.add_assistant_message(assistant_message, tool_calls)
        
        if tool_results:
            for result in tool_results:
                self.add_tool_result(result.get("name", "unknown"), result)
        
        return turn
    
    def _update_context_summary(self):
        """更新上下文摘要"""
        recent_items = list(self.memory_queue)[-10:]
        summaries = []
        
        for item in recent_items:
            if item.type == "user_message":
                summaries.append(f"User: {item.content[:100]}")
            elif item.type == "assistant_message":
                summaries.append(f"Assistant: {item.content[:100]}")
            elif item.type == "tool_result":
                summaries.append(f"Tool: {item.metadata.get('tool_name', 'unknown')}")
        
        self.context_summary = "\n".join(summaries)
    
    def get_recent_context(self, n: int = 10) -> List[MemoryItem]:
        """获取最近的 n 个记忆项"""
        return list(self.memory_queue)[-n:]
    
    def get_conversation_history(self, n: int = 20) -> List[ConversationTurn]:
        """获取最近的对话历史"""
        return self.conversation_history[-n:]
    
    def search_memory(self, query: str) -> List[MemoryItem]:
        """搜索记忆"""
        results = []
        query_lower = query.lower()
        
        for item in self.memory_queue:
            if query_lower in item.content.lower():
                results.append(item)
            elif item.metadata and any(
                query_lower in str(v).lower() 
                for v in item.metadata.values()
            ):
                results.append(item)
        
        return results
    
    def set_preference(self, key: str, value: Any):
        """设置用户偏好"""
        self.user_preferences[key] = value
    
    def get_preference(self, key: str, default: Any = None) -> Any:
        """获取用户偏好"""
        return self.user_preferences.get(key, default)
    
    def clear(self):
        """清除记忆"""
        self.memory_queue.clear()
        self.conversation_history.clear()
        self.context_summary = ""
        self.last_accessed = datetime.now()
    
    def export_to_dict(self) -> Dict[str, Any]:
        """导出为字典"""
        return {
            "session_id": self.session_id,
            "created_at": self.created_at.isoformat(),
            "last_accessed": self.last_accessed.isoformat(),
            "memory_size": len(self.memory_queue),
            "max_size": self.max_size,
            "context_summary": self.context_summary,
            "recent_items": [item.dict() for item in self.get_recent_context()],
            "conversation_turns": len(self.conversation_history),
            "user_preferences": self.user_preferences
        }
    
    def get_context_for_llm(self) -> str:
        """获取用于 LLM 的上下文"""
        context_parts = [
            f"Session ID: {self.session_id}",
            f"Conversation Turns: {len(self.conversation_history)}",
            "\n--- Recent Context ---",
            self.context_summary,
            "\n--- End Context ---"
        ]
        
        return "\n".join(context_parts)


class MemoryManager:
    """记忆管理器 - 管理多个会话"""
    
    def __init__(self):
        self.sessions: Dict[str, ShortTermMemory] = {}
    
    def get_or_create_session(self, session_id: str = None) -> ShortTermMemory:
        """获取或创建会话"""
        if session_id is None:
            session_id = str(uuid.uuid4())
        
        if session_id not in self.sessions:
            self.sessions[session_id] = ShortTermMemory(session_id=session_id)
        
        return self.sessions[session_id]
    
    def get_session(self, session_id: str) -> Optional[ShortTermMemory]:
        """获取会话"""
        return self.sessions.get(session_id)
    
    def delete_session(self, session_id: str) -> bool:
        """删除会话"""
        if session_id in self.sessions:
            del self.sessions[session_id]
            return True
        return False
    
    def list_sessions(self) -> List[str]:
        """列出所有会话"""
        return list(self.sessions.keys())
    
    def cleanup_old_sessions(self, max_age_hours: int = 24):
        """清理旧会话"""
        now = datetime.now()
        to_delete = []
        
        for session_id, session in self.sessions.items():
            age = (now - session.last_accessed).total_seconds() / 3600
            if age > max_age_hours:
                to_delete.append(session_id)
        
        for session_id in to_delete:
            del self.sessions[session_id]
        
        return len(to_delete)


# 全局记忆管理器实例
memory_manager = MemoryManager()
print("✅ ShortTermMemory 和 MemoryManager 创建成功！")

✅ ShortTermMemory 和 MemoryManager 创建成功！


## 工具实现 (工具1)

In [4]:
!pip install httpx python-dotenv

In [5]:
"""
Moodle 工具 - 工具 1（虚拟数据版本）
使用模拟数据，不需要真实 Moodle 连接
符合 MCP 标准
"""

import os
from typing import Dict, Any, Optional
from datetime import datetime, timedelta
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import random

load_dotenv()

class ToolInputSchema(BaseModel):
    type: str = "object"
    properties: Dict[str, Any] = Field(default_factory=dict)
    required: list = Field(default_factory=list)

class ToolDefinition(BaseModel):
    name: str
    description: str
    inputSchema: ToolInputSchema

class MockMoodleTool:
    """虚拟 Moodle API 工具 - 返回模拟数据"""
    
    def __init__(self, moodle_url: str = None, token: str = None, user_id: str = None):
        # 配置信息（仅用于显示，不实际使用）
        self.base_url = moodle_url or os.getenv("MOODLE_URL", "https://moodle.hsu.edu.hk")
        self.token = token or os.getenv("MOODLE_TOKEN", "mock-token")
        self.user_id = user_id or os.getenv("MOODLE_USER_ID", "p253239")
        
        # 生成虚拟数据
        self._mock_data = self._generate_mock_data()
        
        print(f"✅ Moodle 工具已配置（虚拟数据模式）:")
        print(f"   URL: {self.base_url}")
        print(f"   Token: {self.token[:20]}...")
        print(f"   ⚠️  注意：使用虚拟数据，不连接真实 Moodle")
    
    def _generate_mock_data(self) -> Dict[str, Any]:
        """生成虚拟课程数据"""
        return {
            "courses": {
                1: {
                    "name": "人工智能基础",
                    "assignments": [
                        {
                            "id": 101,
                            "name": "作业 1 - Python 编程基础",
                            "duedate": (datetime.now() + timedelta(days=5)).timestamp(),
                            "graded": True,
                            "grade": "85/100",
                            "feedback": "代码结构清晰，但可以优化算法效率。",
                            "submission_status": "submitted",
                            "submit_date": (datetime.now() - timedelta(days=2)).strftime('%Y-%m-%d')
                        },
                        {
                            "id": 102,
                            "name": "作业 2 - 机器学习算法",
                            "duedate": (datetime.now() + timedelta(days=12)).timestamp(),
                            "graded": False,
                            "grade": None,
                            "feedback": None,
                            "submission_status": "submitted",
                            "submit_date": (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
                        },
                        {
                            "id": 103,
                            "name": "作业 3 - 深度学习项目",
                            "duedate": (datetime.now() + timedelta(days=20)).timestamp(),
                            "graded": False,
                            "grade": None,
                            "feedback": None,
                            "submission_status": "not_submitted",
                            "submit_date": None
                        }
                    ],
                    "materials": [
                        {"section": "Week 1", "type": "resource", "title": "课程大纲.pdf", "id": 1001, "url": "#", "date": "2024-01-15"},
                        {"section": "Week 1", "type": "resource", "title": "Python 基础教程.pdf", "id": 1002, "url": "#", "date": "2024-01-16"},
                        {"section": "Week 2", "type": "resource", "title": "机器学习概述.pptx", "id": 1003, "url": "#", "date": "2024-01-22"},
                        {"section": "Week 2", "type": "url", "title": "TensorFlow 官方文档", "id": 1004, "url": "https://tensorflow.org", "date": "2024-01-23"},
                        {"section": "Week 3", "type": "folder", "title": "实验数据", "id": 1005, "url": "#", "date": "2024-01-29"},
                        {"section": "Week 3", "type": "resource", "title": "神经网络基础.pdf", "id": 1006, "url": "#", "date": "2024-01-30"}
                    ],
                    "notes": [
                        {"id": 2001, "title": "Week 3 课程笔记", "type": "resource", "posted_date": "2024-01-29", "is_new": True, "url": "#"},
                        {"id": 2002, "title": "作业 2 补充说明", "type": "resource", "posted_date": "2024-01-28", "is_new": True, "url": "#"},
                        {"id": 2003, "title": "期中考试范围", "type": "resource", "posted_date": "2024-01-25", "is_new": False, "url": "#"}
                    ]
                },
                2: {
                    "name": "数据结构",
                    "assignments": [
                        {
                            "id": 201,
                            "name": "作业 1 - 链表实现",
                            "duedate": (datetime.now() + timedelta(days=3)).timestamp(),
                            "graded": True,
                            "grade": "92/100",
                            "feedback": "优秀的实现！边界条件处理得很好。",
                            "submission_status": "submitted",
                            "submit_date": (datetime.now() - timedelta(days=4)).strftime('%Y-%m-%d')
                        }
                    ],
                    "materials": [
                        {"section": "Week 1", "type": "resource", "title": "链表讲义.pdf", "id": 2001, "url": "#", "date": "2024-01-15"},
                        {"section": "Week 2", "type": "resource", "title": "树结构.pdf", "id": 2002, "url": "#", "date": "2024-01-22"}
                    ],
                    "notes": [
                        {"id": 3001, "title": "实验课安排", "type": "resource", "posted_date": "2024-01-20", "is_new": True, "url": "#"}
                    ]
                }
            }
        }
    
    async def check_assignment_status(self, course_id: int, assignment_id: int = None) -> Dict[str, Any]:
        """虚拟检查作业状态"""
        print(f"🔍 查询课程 {course_id} 的作业状态...")
        
        # 模拟网络延迟
        import asyncio
        await asyncio.sleep(0.5)
        
        course_data = self._mock_data["courses"].get(course_id)
        
        if not course_data:
            return {
                "success": False,
                "error": f"课程 {course_id} 不存在",
                "timestamp": datetime.now().isoformat()
            }
        
        assignments = course_data.get("assignments", [])
        
        if assignment_id:
            assignments = [a for a in assignments if a["id"] == assignment_id]
        
        return {
            "success": True,
            "data": {
                "course_id": course_id,
                "course_name": course_data["name"],
                "assignments": assignments,
                "count": len(assignments)
            },
            "timestamp": datetime.now().isoformat(),
            "tool": "moodle_tool",
            "action": "check_assignment_status"
        }
    
    async def check_new_notes(self, course_id: int) -> Dict[str, Any]:
        """虚拟检查课程笔记/公告"""
        print(f"🔍 查询课程 {course_id} 的新笔记...")
        
        # 模拟网络延迟
        import asyncio
        await asyncio.sleep(0.5)
        
        course_data = self._mock_data["courses"].get(course_id)
        
        if not course_data:
            return {
                "success": False,
                "error": f"课程 {course_id} 不存在",
                "timestamp": datetime.now().isoformat()
            }
        
        notes = course_data.get("notes", [])
        
        return {
            "success": True,
            "data": {
                "course_id": course_id,
                "course_name": course_data["name"],
                "notes": notes,
                "count": len(notes),
                "new_count": sum(1 for n in notes if n.get("is_new", False))
            },
            "timestamp": datetime.now().isoformat(),
            "tool": "moodle_tool",
            "action": "check_new_notes"
        }
    
    async def get_course_materials(self, course_id: int) -> Dict[str, Any]:
        """虚拟获取课程材料"""
        print(f"🔍 获取课程 {course_id} 的材料...")
        
        # 模拟网络延迟
        import asyncio
        await asyncio.sleep(0.5)
        
        course_data = self._mock_data["courses"].get(course_id)
        
        if not course_data:
            return {
                "success": False,
                "error": f"课程 {course_id} 不存在",
                "timestamp": datetime.now().isoformat()
            }
        
        materials = course_data.get("materials", [])
        
        return {
            "success": True,
            "data": {
                "course_id": course_id,
                "course_name": course_data["name"],
                "materials": materials,
                "count": len(materials)
            },
            "timestamp": datetime.now().isoformat(),
            "tool": "moodle_tool",
            "action": "get_course_materials"
        }
    
    def get_tool_definition(self) -> ToolDefinition:
        return ToolDefinition(
            name="moodle_tool",
            description="Access Moodle LMS to check assignments, feedback, notes, and course materials (Mock Data)",
            inputSchema=ToolInputSchema(
                type="object",
                properties={
                    "action": {"type": "string"},
                    "course_id": {"type": "integer"}
                },
                required=["action", "course_id"]
            )
        )

# 创建虚拟 Moodle 工具实例
mock_moodle_tool = MockMoodleTool(
    moodle_url="https://moodle.hsu.edu.hk",
    token="mock-token-for-testing",
    user_id="p253239"
)

print("✅ 虚拟 Moodle 工具已创建！")
print("💡 提示：")
print("   1. 使用课程 ID 1 或 2 测试")
print("   2. 数据是虚拟的，不会连接真实 Moodle")
print("   3. 适合演示和开发测试")

✅ Moodle 工具已配置（虚拟数据模式）:
   URL: https://moodle.hsu.edu.hk
   Token: mock-token-for-testi...
   ⚠️  注意：使用虚拟数据，不连接真实 Moodle
✅ 虚拟 Moodle 工具已创建！
💡 提示：
   1. 使用课程 ID 1 或 2 测试
   2. 数据是虚拟的，不会连接真实 Moodle
   3. 适合演示和开发测试


## 工具实现 (工具2)

In [6]:
"""
总结工具 - 工具 2
使用 qwen3:0.6b 模型总结课程材料
符合 MCP 标准
"""
# 总结工具 - 工具 2（独立版本）
# 使用 qwen3:0.6b 模型总结课程材料

from typing import Dict, List, Any, Optional
import os
from datetime import datetime
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# 加载环境变量
load_dotenv()

# 直接在当前文件定义 MCP 相关的类
class ToolInputSchema(BaseModel):
    type: str = "object"
    properties: Dict[str, Any] = Field(default_factory=dict)
    required: List[str] = Field(default_factory=list)

class ToolDefinition(BaseModel):
    name: str
    description: str
    inputSchema: ToolInputSchema

class SummaryTool:
    """AI 总结工具 - 使用 qwen3:0.6b 模型"""
    
    def __init__(self):
        self.ollama_host = os.getenv("OLLAMA_HOST", "http://localhost:11434")
        self.model = os.getenv("OLLAMA_MODEL", "qwen3:0.6b")
    
    async def summarize_content(
        self, 
        content: str, 
        max_length: int = 500,
        language: str = "english"
    ) -> Dict[str, Any]:
        """使用 qwen3:0.6b 模型总结给定内容"""
        try:
            # 尝试调用 qwen3:0.6b 模型
            try:
                import ollama
                
                prompt = f"""Summarize the following content in {language}, maximum {max_length} words. 
Focus on key points and main ideas:

{content}

Summary:"""
                
                response = ollama.chat(
                    model=self.model,
                    messages=[
                        {
                            "role": "system",
                            "content": "You are a helpful assistant that summarizes educational content clearly and concisely."
                        },
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ]
                )
                
                summary = response['message']['content']
                
            except Exception as e:
                # 如果 ollama 不可用，使用简单总结
                summary = self._simple_summary(content, max_length)
            
            return {
                "success": True,
                "summary": summary,
                "original_length": len(content),
                "summary_length": len(summary),
                "compression_ratio": round(len(summary) / len(content) * 100, 2) if content else 0,
                "model": self.model,
                "timestamp": datetime.now().isoformat(),
                "tool": "summary_tool",
                "action": "summarize_content"
            }
            
        except Exception as e:
            return {
                "success": False,
                "error": str(e),
                "timestamp": datetime.now().isoformat()
            }
    
    def _simple_summary(self, content: str, max_length: int) -> str:
        """简单总结（降级方案）"""
        if not content:
            return "No content to summarize."
        
        # 提取关键句子
        sentences = content.replace('\n', ' ').split('.')
        key_sentences = sentences[:min(5, len(sentences))]
        summary = '. '.join(key_sentences) + '.'
        
        return summary[:max_length]
    
    async def summarize_module(
        self, 
        module_name: str, 
        materials: List[Dict[str, Any]]
    ) -> Dict[str, Any]:
        """总结整个模块的材料"""
        try:
            # 合并所有材料内容
            all_content = "\n\n".join([
                f"{m.get('title', 'Untitled')} ({m.get('type', 'unknown')}): {m.get('content', '') or m.get('title', '')}" 
                for m in materials
            ])
            
            summary_result = await self.summarize_content(all_content, max_length=1000)
            
            if summary_result["success"]:
                return {
                    "success": True,
                    "module_name": module_name,
                    "summary": summary_result["summary"],
                    "materials_count": len(materials),
                    "model": self.model,
                    "timestamp": datetime.now().isoformat(),
                    "tool": "summary_tool",
                    "action": "summarize_module"
                }
            else:
                return summary_result
                
        except Exception as e:
            return {
                "success": False,
                "error": str(e),
                "timestamp": datetime.now().isoformat()
            }
    
    def get_tool_definition(self) -> ToolDefinition:
        """获取工具定义"""
        return ToolDefinition(
            name="summary_tool",
            description="Summarize course materials and content using qwen3:0.6b AI model",
            inputSchema=ToolInputSchema(
                type="object",
                properties={
                    "action": {
                        "type": "string",
                        "enum": ["summarize_content", "summarize_module"],
                        "description": "Summary action to perform"
                    },
                    "content": {
                        "type": "string",
                        "description": "Content to summarize"
                    },
                    "module_name": {
                        "type": "string",
                        "description": "Module name"
                    },
                    "materials": {
                        "type": "array",
                        "items": {"type": "object"},
                        "description": "List of materials"
                    },
                    "max_length": {
                        "type": "integer",
                        "default": 500
                    },
                    "language": {
                        "type": "string",
                        "default": "english",
                        "enum": ["english", "chinese"]
                    }
                },
                required=["action"]
            )
        )

# 创建工具实例
summary_tool = SummaryTool()
print("✅ SummaryTool 创建成功！")
print(f"模型：{summary_tool.model}")

✅ SummaryTool 创建成功！
模型：qwen3:0.6b


## Agent 核心

In [7]:
"""
AI Agent 核心
实现多步推理、规划和工具利用
使用 qwen3:0.6b 模型
"""
# AI Agent 核心 - 独立版本
# 实现多步推理、规划和工具利用
# 使用 qwen3:0.6b 模型

from typing import Dict, List, Any, Optional
import asyncio
import json
import uuid
from datetime import datetime
from collections import deque

# 如果已安装 ollama，尝试导入
try:
    import ollama
    OLLAMA_AVAILABLE = True
except ImportError:
    OLLAMA_AVAILABLE = False
    print("⚠️  警告：ollama 未安装，将使用降级方案")

# 定义所需的类（如果之前没有定义）
try:
    from src.mcp.schema import MemoryItem, ConversationTurn
except ImportError:
    from pydantic import BaseModel, Field
    
    class MemoryItem(BaseModel):
        id: str
        timestamp: str
        content: str
        type: str
        metadata: Dict[str, Any] = Field(default_factory=dict)
    
    class ConversationTurn(BaseModel):
        turn_id: str
        user_message: str
        assistant_message: Optional[str] = None
        tool_calls: List = Field(default_factory=list)
        tool_results: List = Field(default_factory=list)
        timestamp: str = Field(default_factory=lambda: datetime.now().isoformat())


class ShortTermMemory:
    """短期记忆管理器"""
    
    def __init__(self, max_size: int = 50, session_id: Optional[str] = None):
        self.session_id = session_id or str(uuid.uuid4())
        self.max_size = max_size
        self.memory_queue: deque = deque(maxlen=max_size)
        self.conversation_history: List[ConversationTurn] = []
        self.context_summary: str = ""
        self.created_at = datetime.now()
        self.last_accessed = datetime.now()
        self.user_preferences: Dict[str, Any] = {}
    
    def add_item(self, content: str, item_type: str, metadata: Dict[str, Any] = None) -> MemoryItem:
        item = MemoryItem(
            id=str(uuid.uuid4()),
            timestamp=datetime.now().isoformat(),
            content=content,
            type=item_type,
            metadata=metadata or {}
        )
        self.memory_queue.append(item)
        self.last_accessed = datetime.now()
        self._update_context_summary()
        return item
    
    def add_user_message(self, message: str) -> MemoryItem:
        return self.add_item(message, "user_message")
    
    def add_assistant_message(self, message: str, tool_calls: List = None) -> MemoryItem:
        metadata = {"tool_calls": tool_calls} if tool_calls else {}
        return self.add_item(message, "assistant_message", metadata)
    
    def add_tool_result(self, tool_name: str, result: Dict) -> MemoryItem:
        content = f"Tool '{tool_name}' returned: {json.dumps(result, ensure_ascii=False)[:500]}"
        return self.add_item(content, "tool_result", {"tool_name": tool_name, "result": result})
    
    def _update_context_summary(self):
        recent_items = list(self.memory_queue)[-10:]
        summaries = []
        for item in recent_items:
            if item.type == "user_message":
                summaries.append(f"User: {item.content[:100]}")
            elif item.type == "assistant_message":
                summaries.append(f"Assistant: {item.content[:100]}")
            elif item.type == "tool_result":
                summaries.append(f"Tool: {item.metadata.get('tool_name', 'unknown')}")
        self.context_summary = "\n".join(summaries)
    
    def get_recent_context(self, n: int = 10) -> List[MemoryItem]:
        return list(self.memory_queue)[-n:]
    
    def clear(self):
        self.memory_queue.clear()
        self.conversation_history.clear()
        self.context_summary = ""
        self.last_accessed = datetime.now()
    
    def export_to_dict(self) -> Dict[str, Any]:
        return {
            "session_id": self.session_id,
            "created_at": self.created_at.isoformat(),
            "last_accessed": self.last_accessed.isoformat(),
            "memory_size": len(self.memory_queue),
            "max_size": self.max_size,
            "context_summary": self.context_summary,
            "recent_items": [item.dict() for item in self.get_recent_context()],
            "conversation_turns": len(self.conversation_history)
        }


class MemoryManager:
    """记忆管理器"""
    
    def __init__(self):
        self.sessions: Dict[str, ShortTermMemory] = {}
    
    def get_or_create_session(self, session_id: str = None) -> ShortTermMemory:
        if session_id is None:
            session_id = str(uuid.uuid4())
        if session_id not in self.sessions:
            self.sessions[session_id] = ShortTermMemory(session_id=session_id)
        return self.sessions[session_id]
    
    def get_session(self, session_id: str) -> Optional[ShortTermMemory]:
        return self.sessions.get(session_id)
    
    def delete_session(self, session_id: str) -> bool:
        if session_id in self.sessions:
            del self.sessions[session_id]
            return True
        return False


# 全局记忆管理器
memory_manager = MemoryManager()


# 更新 Agent 使用虚拟 Moodle 工具
class MockAIAgent:
    """使用虚拟 Moodle 的 AI Agent"""
    
    def __init__(self):
        self.moodle_tool = mock_moodle_tool
        self.session_id = "session_mock_moodle"
    
    async def process_message(self, user_message: str) -> Dict[str, Any]:
        message_lower = user_message.lower()
        response = ""
        
        # 提取课程 ID
        import re
        course_match = re.search(r'(?:课程|course|id[:\s])\s*(\d+)', message_lower)
        course_id = int(course_match.group(1)) if course_match else 1
        
        if "作业" in message_lower or "assignment" in message_lower:
            result = await self.moodle_tool.check_assignment_status(course_id)
            
            if result["success"] and result["data"]["assignments"]:
                response = f"📋 课程 {course_id} ({result['data']['course_name']}) 的作业状态:\n\n"
                for assign in result["data"]["assignments"][:5]:
                    response += f"📝 {assign['name']}\n"
                    response += f"  - 状态：{assign['submission_status']}\n"
                    if assign['graded']:
                        response += f"  - 评分：{assign['grade']}\n"
                        if assign['feedback']:
                            response += f"  - 反馈：{assign['feedback']}\n"
                    if assign['duedate']:
                        due_date = datetime.fromtimestamp(assign['duedate']).strftime('%Y-%m-%d')
                        response += f"  - 截止日期：{due_date}\n"
                    response += "\n"
            else:
                response = f"❌ 无法获取课程 {course_id} 的作业信息\n"
                if not result["success"]:
                    response += f"错误：{result.get('error', '未知错误')}"
        
        elif "笔记" in message_lower or "notes" in message_lower or "材料" in message_lower:
            result = await self.moodle_tool.get_course_materials(course_id)
            
            if result["success"] and result["data"]["materials"]:
                response = f"📚 课程 {course_id} ({result['data']['course_name']}) 的材料 ({result['data']['count']} 项):\n\n"
                
                # 按类型分组
                by_type = {}
                for mat in result["data"]["materials"]:
                    mat_type = mat["type"]
                    if mat_type not in by_type:
                        by_type[mat_type] = []
                    by_type[mat_type].append(mat)
                
                for mat_type, items in by_type.items():
                    response += f"📁 {mat_type.upper()} ({len(items)}):\n"
                    for item in items[:5]:
                        response += f"  • {item['title']}\n"
                    if len(items) > 5:
                        response += f"  ... 还有 {len(items)-5} 项\n"
                    response += "\n"
            else:
                response = f"❌ 无法获取课程 {course_id} 的材料\n"
        
        else:
            response = """💡 我可以帮您：
- 检查作业：输入 "课程 1 的作业"
- 查看材料：输入 "课程 1 的材料"
- 查看笔记：输入 "课程 1 的笔记"

可用课程 ID: 1 (人工智能基础), 2 (数据结构)

请告诉我您需要什么帮助？"""
        
        return {
            "success": True,
            "response": response,
            "session_id": self.session_id
        }

# 创建虚拟 Agent
mock_agent = MockAIAgent()
print("✅ 虚拟 AI Agent 已创建！")

class SimpleSummaryTool:
    """简化的总结工具"""
    
    def __init__(self):
        self.model = "qwen3:0.6b"
    
    async def summarize_content(self, content: str, max_length: int = 500, language: str = "english") -> Dict[str, Any]:
        if OLLAMA_AVAILABLE:
            try:
                prompt = f"Summarize the following in {language}, max {max_length} words:\n{content}"
                response = ollama.chat(
                    model=self.model,
                    messages=[{"role": "user", "content": prompt}]
                )
                summary = response['message']['content']
            except:
                summary = self._simple_summary(content, max_length)
        else:
            summary = self._simple_summary(content, max_length)
        
        return {
            "success": True,
            "summary": summary,
            "original_length": len(content),
            "summary_length": len(summary),
            "model": self.model,
            "timestamp": datetime.now().isoformat()
        }
    
    def _simple_summary(self, content: str, max_length: int) -> str:
        sentences = content.replace('\n', ' ').split('.')
        key_sentences = sentences[:min(5, len(sentences))]
        return '. '.join(key_sentences)[:max_length]
    
    async def summarize_module(self, module_name: str, materials: List[Dict[str, Any]]) -> Dict[str, Any]:
        all_content = "\n\n".join([m.get('title', '') for m in materials])
        summary_result = await self.summarize_content(all_content, max_length=1000)
        return {
            "success": True,
            "module_name": module_name,
            "summary": summary_result["summary"],
            "materials_count": len(materials),
            "timestamp": datetime.now().isoformat()
        }



class AIAgent:
    """自主 AI Agent - 使用 qwen3:0.6b 模型"""
    
    def __init__(self, session_id: str = None):
        self.session_id = session_id or str(uuid.uuid4())
        self.memory = memory_manager.get_or_create_session(self.session_id)
        self.tools = {
            "moodle_tool": moodle_tool,
            "summary_tool": summary_tool
        }
        self.model = "qwen3:0.6b"
    
    async def process_message(self, user_message: str) -> Dict[str, Any]:
        """处理用户消息"""
        try:
            # 1. 记录用户消息
            self.memory.add_user_message(user_message)
            
            # 2. 分析意图并规划
            plan = await self._analyze_and_plan(user_message)
            
            # 3. 执行计划
            results = await self._execute_plan(plan)
            
            # 4. 生成响应
            response = await self._generate_response(user_message, plan, results)
            
            # 5. 记录助手响应
            self.memory.add_assistant_message(response)
            
            return {
                "success": True,
                "response": response,
                "plan": plan,
                "tool_results": results,
                "session_id": self.session_id,
                "model": self.model
            }
            
        except Exception as e:
            error_response = f"Sorry, I encountered an error: {str(e)}"
            self.memory.add_assistant_message(error_response)
            return {
                "success": False,
                "response": error_response,
                "error": str(e),
                "session_id": self.session_id
            }
    
    async def _analyze_and_plan(self, user_message: str) -> Dict[str, Any]:
        """分析用户意图并创建执行计划"""
        plan = {
            "steps": [],
            "tools_needed": [],
            "estimated_steps": 0
        }
        
        message_lower = user_message.lower()
        
        # 检查作业状态
        if any(word in message_lower for word in ["assignment", "作业", "评分", "feedback", "反馈", "grade"]):
            plan["steps"].append({
                "action": "check_assignment",
                "tool": "moodle_tool",
                "description": "Check assignment status and feedback"
            })
            plan["tools_needed"].append("moodle_tool")
        
        # 检查新笔记
        if any(word in message_lower for word in ["notes", "笔记", "通知", "new", "新"]):
            plan["steps"].append({
                "action": "check_notes",
                "tool": "moodle_tool",
                "description": "Check for new course notes"
            })
            plan["tools_needed"].append("moodle_tool")
        
        # 总结材料
        if any(word in message_lower for word in ["summarize", "总结", "materials", "材料", "module", "模块"]):
            plan["steps"].append({
                "action": "get_materials",
                "tool": "moodle_tool",
                "description": "Get course materials"
            })
            plan["steps"].append({
                "action": "summarize_module",
                "tool": "summary_tool",
                "description": "Summarize the materials"
            })
            plan["tools_needed"].extend(["moodle_tool", "summary_tool"])
        
        if not plan["steps"]:
            plan["steps"].append({
                "action": "general_query",
                "tool": None,
                "description": "Handle general query"
            })
        
        plan["estimated_steps"] = len(plan["steps"])
        return plan
    
    async def _execute_plan(self, plan: Dict[str, Any]) -> List[Dict]:
        """执行计划中的步骤"""
        results = []
        
        for step in plan["steps"]:
            tool_name = step.get("tool")
            action = step.get("action")
            
            if tool_name and tool_name in self.tools:
                tool = self.tools[tool_name]
                
                try:
                    if action == "check_assignment":
                        result = await tool.check_assignment_status(course_id=101)
                    elif action == "check_notes":
                        result = await tool.check_new_notes(course_id=101)
                    elif action == "get_materials":
                        result = await tool.get_course_materials(course_id=101)
                    elif action == "summarize_module":
                        materials_result = await moodle_tool.get_course_materials(course_id=101)
                        if materials_result["success"]:
                            materials = materials_result["data"].get("materials", [])
                            result = await tool.summarize_module(
                                module_name="AI Course",
                                materials=materials
                            )
                        else:
                            result = materials_result
                    else:
                        result = {"success": True, "data": "General query handled"}
                    
                    self.memory.add_tool_result(tool_name, result)
                    results.append({
                        "step": action,
                        "tool": tool_name,
                        "result": result,
                        "success": result.get("success", False)
                    })
                    
                except Exception as e:
                    results.append({
                        "step": action,
                        "tool": tool_name,
                        "result": {"success": False, "error": str(e)},
                        "success": False
                    })
            else:
                results.append({
                    "step": action,
                    "tool": None,
                    "result": {"success": True, "data": "No tool needed"},
                    "success": True
                })
        
        return results
    
    async def _generate_response(self, user_message: str, plan: Dict[str, Any], results: List[Dict]) -> str:
        """生成自然语言响应"""
        response_parts = []
        
        for result in results:
            if result["success"]:
                step = result["step"]
                tool_result = result["result"]
                
                if step == "check_assignment":
                    data = tool_result.get("data", {})
                    assignments = data.get("assignments", [])
                    if assignments:
                        assign = assignments[0]
                        response_parts.append(
                            f"📋 **作业状态**:\n"
                            f"- 名称：{assign.get('name', 'N/A')}\n"
                            f"- 评分：{assign.get('grade', '未评分')}\n"
                            f"- 反馈：{assign.get('feedback', '无反馈')}\n"
                            f"- 提交状态：{assign.get('submission_status', '未知')}"
                        )
                    else:
                        response_parts.append("未找到作业信息。")
                
                elif step == "check_notes":
                    data = tool_result.get("data", {})
                    notes = data.get("notes", [])
                    if notes:
                        response_parts.append(f"📝 **新笔记** ({len(notes)} 条):")
                        for note in notes[:3]:
                            response_parts.append(
                                f"  • {note.get('title', 'Untitled')} - {note.get('posted_date', '')}"
                            )
                    else:
                        response_parts.append("暂无新笔记。")
                
                elif step == "summarize_module":
                    summary = tool_result.get("summary", "无法生成总结")
                    response_parts.append(f"📚 **模块总结**:\n{summary}")
                
                elif step == "get_materials":
                    data = tool_result.get("data", {})
                    materials = data.get("materials", [])
                    response_parts.append(f"📖 **课程材料** ({len(materials)} 项):")
                    for mat in materials[:5]:
                        response_parts.append(f"  • [{mat.get('type', '')}] {mat.get('title', '')}")
            
            else:
                response_parts.append(f"⚠️ 步骤 '{result['step']}' 执行失败：{result['result'].get('error', '未知错误')}")
        
        if not response_parts:
            response_parts.append("您好！我可以帮助您检查作业状态、查看新笔记或总结课程材料。请告诉我您需要什么帮助？")
        
        return "\n\n".join(response_parts)
    
    def get_memory_status(self) -> Dict[str, Any]:
        """获取记忆状态"""
        return self.memory.export_to_dict()
    
    def clear_memory(self):
        """清除记忆"""
        self.memory.clear()
    
    async def chat(self, message: str) -> str:
        """简单的聊天接口"""
        result = await self.process_message(message)
        return result.get("response", "Error occurred")


# 使用虚拟工具
moodle_tool = mock_moodle_tool
summary_tool = SimpleSummaryTool()

# 更新 AIAgent 类使用虚拟工具
agent = AIAgent()
print("✅ AIAgent 创建成功！")
print(f"模型：{agent.model}")
print(f"会话 ID: {agent.session_id}")
print(f"⚠️  注意：使用虚拟 Moodle 数据")

✅ 虚拟 AI Agent 已创建！
✅ AIAgent 创建成功！
模型：qwen3:0.6b
会话 ID: 3d517aad-dff7-45bf-8718-09fa72fbf211
⚠️  注意：使用虚拟 Moodle 数据


## 对话界面

In [8]:
!pip install nest_asyncio

In [9]:
!pip install fastapi uvicorn websockets

In [15]:
# 更新 Agent 使用虚拟 Moodle 工具
class MockAIAgent:
    """使用虚拟 Moodle 的 AI Agent"""
    
    def __init__(self):
        self.moodle_tool = mock_moodle_tool
        self.session_id = "session_mock_moodle"
    
    async def process_message(self, user_message: str) -> Dict[str, Any]:
        message_lower = user_message.lower()
        response = ""
        
        # 提取课程 ID
        import re
        course_match = re.search(r'(?:课程|course|id[:\s])\s*(\d+)', message_lower)
        course_id = int(course_match.group(1)) if course_match else 1
        
        if "作业" in message_lower or "assignment" in message_lower:
            result = await self.moodle_tool.check_assignment_status(course_id)
            
            if result["success"] and result["data"]["assignments"]:
                response = f"📋 课程 {course_id} ({result['data']['course_name']}) 的作业状态:\n\n"
                for assign in result["data"]["assignments"][:5]:
                    response += f"📝 {assign['name']}\n"
                    response += f"  - 状态：{assign['submission_status']}\n"
                    if assign['graded']:
                        response += f"  - 评分：{assign['grade']}\n"
                        if assign['feedback']:
                            response += f"  - 反馈：{assign['feedback']}\n"
                    if assign['duedate']:
                        due_date = datetime.fromtimestamp(assign['duedate']).strftime('%Y-%m-%d')
                        response += f"  - 截止日期：{due_date}\n"
                    response += "\n"
            else:
                response = f"❌ 无法获取课程 {course_id} 的作业信息\n"
                if not result["success"]:
                    response += f"错误：{result.get('error', '未知错误')}"
        
        elif "笔记" in message_lower or "notes" in message_lower or "材料" in message_lower:
            result = await self.moodle_tool.get_course_materials(course_id)
            
            if result["success"] and result["data"]["materials"]:
                response = f"📚 课程 {course_id} ({result['data']['course_name']}) 的材料 ({result['data']['count']} 项):\n\n"
                
                # 按类型分组
                by_type = {}
                for mat in result["data"]["materials"]:
                    mat_type = mat["type"]
                    if mat_type not in by_type:
                        by_type[mat_type] = []
                    by_type[mat_type].append(mat)
                
                for mat_type, items in by_type.items():
                    response += f"📁 {mat_type.upper()} ({len(items)}):\n"
                    for item in items[:5]:
                        response += f"  • {item['title']}\n"
                    if len(items) > 5:
                        response += f"  ... 还有 {len(items)-5} 项\n"
                    response += "\n"
            else:
                response = f"❌ 无法获取课程 {course_id} 的材料\n"
        
        else:
            response = """💡 我可以帮您：
- 检查作业：输入 "课程 1 的作业"
- 查看材料：输入 "课程 1 的材料"
- 查看笔记：输入 "课程 1 的笔记"

可用课程 ID: 1 (人工智能基础), 2 (数据结构)

请告诉我您需要什么帮助？"""
        
        return {
            "success": True,
            "response": response,
            "session_id": self.session_id
        }

# 创建虚拟 Agent
mock_agent = MockAIAgent()
print("✅ 虚拟 AI Agent 已创建！")

✅ 虚拟 AI Agent 已创建！


## 测试文件

In [16]:
# 测试代码 - 使用虚拟数据
import pytest
import asyncio
from typing import Dict, Any
from datetime import datetime

# 使用虚拟 Moodle 工具
moodle_tool = mock_moodle_tool

# 运行测试
print("🧪 开始运行测试（虚拟数据模式）...\n")

# 测试 1: Moodle 工具 - 检查作业
async def test_moodle_check_assignment():
    print("Test 1: Moodle 工具 - 检查作业状态")
    result = await moodle_tool.check_assignment_status(course_id=1)
    assert result["success"] == True
    assert "data" in result
    assert "timestamp" in result
    assert result["tool"] == "moodle_tool"
    print(f"✅ 通过: 找到 {result['data']['count']} 个作业\n")

# 测试 2: Moodle 工具 - 检查笔记
async def test_moodle_check_notes():
    print("Test 2: Moodle 工具 - 检查新笔记")
    result = await moodle_tool.check_new_notes(course_id=1)
    assert result["success"] == True
    assert "notes" in result["data"]
    assert result["tool"] == "moodle_tool"
    print(f"✅ 通过: 找到 {len(result['data']['notes'])} 条笔记\n")

# 测试 3: 总结工具 - 总结内容
async def test_summary_content():
    print("Test 3: 总结工具 - 总结内容")
    content = """
    This is a sample course material about Machine Learning.
    It covers supervised learning, unsupervised learning, and reinforcement learning.
    Key algorithms include linear regression, decision trees, and neural networks.
    """
    result = await summary_tool.summarize_content(content, max_length=200)
    assert result["success"] == True
    assert "summary" in result
    assert result["tool"] == "summary_tool"
    print(f"✅ 通过: 压缩率 {result['compression_ratio']}%\n")

# 测试 4: MCP 工具定义
async def test_mcp_definitions():
    print("Test 4: MCP 工具定义验证")
    moodle_def = moodle_tool.get_tool_definition()
    summary_def = summary_tool.get_tool_definition()
    
    assert moodle_def.name == "moodle_tool"
    assert moodle_def.inputSchema.type == "object"
    assert "action" in moodle_def.inputSchema.required
    
    assert summary_def.name == "summary_tool"
    assert summary_def.inputSchema.type == "object"
    assert "action" in summary_def.inputSchema.required
    
    print("✅ 通过：MCP 工具定义有效\n")

# 测试 5: 获取课程材料
async def test_get_materials():
    print("Test 5: Moodle 工具 - 获取课程材料")
    result = await moodle_tool.get_course_materials(course_id=1)
    assert result["success"] == True
    assert "materials" in result["data"]
    assert result["data"]["count"] > 0
    print(f"✅ 通过: 找到 {result['data']['count']} 个材料\n")

# 运行所有测试
async def run_all_tests():
    try:
        await test_moodle_check_assignment()
        await test_moodle_check_notes()
        await test_summary_content()
        await test_mcp_definitions()
        await test_get_materials()
        
        print("=" * 50)
        print("✅ 所有测试通过！")
        print("=" * 50)
    except AssertionError as e:
        print(f"❌ 测试失败：{e}")
    except Exception as e:
        print(f"❌ 测试出错：{e}")

# 执行测试
await run_all_tests()

🧪 开始运行测试（虚拟数据模式）...

Test 1: Moodle 工具 - 检查作业状态
🔍 查询课程 1 的作业状态...
✅ 通过: 找到 3 个作业

Test 2: Moodle 工具 - 检查新笔记
🔍 查询课程 1 的新笔记...
✅ 通过: 找到 3 条笔记

Test 3: 总结工具 - 总结内容
❌ 测试出错：'tool'


## 主入口文件

In [17]:
"""
主入口文件
启动 MCP 服务器和对话界面
"""
# main.py - Jupyter 适配版本
import sys
import os
import asyncio
import multiprocessing

# 添加 src 到路径
try:
    # 如果是脚本文件
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # 如果在 Jupyter Notebook 中
    current_dir = os.getcwd()
    print("✅ 检测到 Jupyter 环境")

if current_dir not in sys.path:
    sys.path.insert(0, current_dir)
    print(f"✅ 已添加路径: {current_dir}")

# 验证 src 目录
src_dir = os.path.join(current_dir, 'src')
if os.path.exists(src_dir) and src_dir not in sys.path:
    sys.path.insert(0, src_dir)
    print(f"✅ src 目录已添加: {src_dir}")

def run_mcp_server():
    """运行 MCP 服务器"""
    from src.mcp.server import mcp_server
    from src.tools.moodle_tool import moodle_tool
    from src.tools.summary_tool import summary_tool
    
    # 注册工具
    mcp_server.register_tool(
        "moodle_tool",
        moodle_tool.check_assignment_status,
        moodle_tool.get_tool_definition()
    )
    
    mcp_server.register_tool(
        "summary_tool",
        summary_tool.summarize_content,
        summary_tool.get_tool_definition()
    )
    
    # 启动服务器
    mcp_server.run(host="0.0.0.0", port=8000)

def run_chat_interface():
    """运行对话界面"""
    from src.interface.chat_interface import run_interface
    run_interface(host="0.0.0.0", port=8080)

def main():
    """主函数"""
    print("=" * 60)
    print("🤖 Moodle AI Agent 启动中...")
    print("=" * 60)
    print()
    
    # 创建多进程
    p1 = multiprocessing.Process(target=run_mcp_server)
    p2 = multiprocessing.Process(target=run_chat_interface)
    
    p1.start()
    p2.start()
    
    print("\n✅ MCP Server: http://localhost:8000")
    print("✅ Chat Interface: http://localhost:8080")
    print()
    print("📋 可用功能:")
    print("   1. 检查作业评分和教师反馈")
    print("   2. 查看新课程笔记")
    print("   3. 总结课程材料")
    print()
    print("🔧 MCP 工具:")
    print("   - moodle_tool (Moodle API 访问)")
    print("   - summary_tool (AI 内容总结)")
    print()
    print("🤖 AI 模型：qwen3:0.6b")
    print()
    print("⚠️  按 Ctrl+C 停止服务器")
    print("=" * 60)
    
    try:
        p1.join()
        p2.join()
    except KeyboardInterrupt:
        print("\n🛑 正在停止服务器...")
        p1.terminate()
        p2.terminate()
        p1.join()
        p2.join()
        print("✅ 服务器已停止")

# 运行主函数
if __name__ == "__main__":
    main()

✅ 检测到 Jupyter 环境
🤖 Moodle AI Agent 启动中...


✅ MCP Server: http://localhost:8000
✅ Chat Interface: http://localhost:8080

📋 可用功能:
   1. 检查作业评分和教师反馈
   2. 查看新课程笔记
   3. 总结课程材料

🔧 MCP 工具:
   - moodle_tool (Moodle API 访问)
   - summary_tool (AI 内容总结)

🤖 AI 模型：qwen3:0.6b

⚠️  按 Ctrl+C 停止服务器


## 连接测试

In [18]:
# 测试虚拟 Moodle 连接
import asyncio

async def test_mock_moodle():
    print("🧪 测试虚拟 Moodle 连接...\n")
    
    # 测试 1：获取课程材料
    print("测试 1：获取课程材料")
    result = await mock_moodle_tool.get_course_materials(course_id=1)
    
    if result["success"]:
        print(f"✅ 成功获取 {result['data']['count']} 个材料")
        if result['data']['materials']:
            print(f"   示例：{result['data']['materials'][0]['title']}")
    else:
        print(f"❌ 失败：{result.get('error')}")
    
    # 测试 2：检查作业
    print("\n测试 2：检查作业")
    result = await mock_moodle_tool.check_assignment_status(course_id=1)
    
    if result["success"]:
        print(f"✅ 成功获取 {result['data']['count']} 个作业")
        for assign in result['data']['assignments'][:2]:
            print(f"   - {assign['name']}: {assign['submission_status']}")
    else:
        print(f"❌ 失败：{result.get('error')}")
    
    # 测试 3：检查笔记
    print("\n测试 3：检查新笔记")
    result = await mock_moodle_tool.check_new_notes(course_id=1)
    
    if result["success"]:
        print(f"✅ 成功获取 {result['data']['count']} 条笔记")
        print(f"   新笔记：{result['data']['new_count']} 条")
    else:
        print(f"❌ 失败：{result.get('error')}")
    
    print("\n" + "=" * 50)
    print("✅ 虚拟 Moodle 工具运行正常！")
    print("=" * 50)

# 运行测试
await test_mock_moodle()

🧪 测试虚拟 Moodle 连接...

测试 1：获取课程材料
🔍 获取课程 1 的材料...
✅ 成功获取 6 个材料
   示例：课程大纲.pdf

测试 2：检查作业
🔍 查询课程 1 的作业状态...
✅ 成功获取 3 个作业
   - 作业 1 - Python 编程基础: submitted
   - 作业 2 - 机器学习算法: submitted

测试 3：检查新笔记
🔍 查询课程 1 的新笔记...
✅ 成功获取 3 条笔记
   新笔记：2 条

✅ 虚拟 Moodle 工具运行正常！


In [22]:
from IPython.display import display, HTML, Javascript
import asyncio

# 创建简单的对话界面
html_code = """
<div style="font-family: Arial; max-width: 800px; margin: 0 auto; padding: 20px;">
    <div style="background: white; padding: 30px; border-radius: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.1);">
        <h1 style="color: #667eea; text-align: center;">🤖 Moodle AI Agent</h1>
        <div style="text-align: center; color: #28a745; margin: 10px 0;">✅ 就绪</div>
        
        <div id="chatBox" style="height: 400px; overflow-y: auto; border: 1px solid #ddd; padding: 15px; margin: 20px 0; background: #f9f9f9; border-radius: 5px;">
            <div style="margin: 10px 0; padding: 10px; border-radius: 5px; background: #e0e0e0; margin-right: 20%;">
                您好！我是您的 Moodle AI 助手。请输入：
                <br>• "检查作业" - 查看作业评分
                <br>• "查看笔记" - 查看新笔记
                <br>• "总结材料" - 总结课程内容
            </div>
        </div>
        
        <div style="display: flex; gap: 10px;">
            <input type="text" id="msgInput" placeholder="输入您的问题..." 
                   style="flex: 1; padding: 12px; border: 2px solid #ddd; border-radius: 5px;" 
                   onkeypress="if(event.key==='Enter')handleSend()">
            <button onclick="handleSend()" 
                    style="padding: 12px 30px; background: #667eea; color: white; border: none; border-radius: 5px; cursor: pointer;">
                发送
            </button>
        </div>
    </div>
</div>

<script>
function handleSend() {
    const input = document.getElementById('msgInput');
    const chatBox = document.getElementById('chatBox');
    const msg = input.value.trim();
    if (!msg) return;
    
    // 添加用户消息
    chatBox.innerHTML += `<div style="margin: 10px 0; padding: 10px; border-radius: 5px; background: #667eea; color: white; margin-left: 20%;">${msg}</div>`;
    input.value = '';
    chatBox.scrollTop = chatBox.scrollHeight;
    
    // 模拟响应
    setTimeout(() => {
        let response = '';
        const lowerMsg = msg.toLowerCase();
        
        if (lowerMsg.includes('作业') || lowerMsg.includes('assignment')) {
            response = '📋 **作业状态**:<br>- 名称：Assignment 1<br>- 评分：85/100<br>- 反馈：Good work!';
        } else if (lowerMsg.includes('笔记') || lowerMsg.includes('notes')) {
            response = '📝 **新笔记** (2 条):<br>• Week 3 Notes - 2024-01-14<br>• Assignment Guidelines - 2024-01-13';
        } else if (lowerMsg.includes('总结') || lowerMsg.includes('summarize')) {
            response = '📚 **模块总结**:<br>主要内容涵盖机器学习基础、神经网络、优化算法...';
        } else {
            response = '您好！我可以帮您检查作业、查看笔记或总结材料。';
        }
        
        chatBox.innerHTML += `<div style="margin: 10px 0; padding: 10px; border-radius: 5px; background: #e0e0e0; margin-right: 20%;">${response}</div>`;
        chatBox.scrollTop = chatBox.scrollHeight;
    }, 500);
}
</script>
"""

display(HTML(html_code))
print("✅ 对话界面已显示在上方！")

✅ 对话界面已显示在上方！
